# Lab 1 · Exercise B — LHCb mass peak and the central limit theorem
**B1.** Simplified dataset modelled on an LHCb analysis: histogram of a B-meson invariant mass (`examples/lhcb-b0-ppbar/data/b0_invariant_mass.root`, histogram `massaB0`).
**B2.** Sum of N uniform variables: where does the Gaussian approximation stop working?

In [1]:
import ROOT
#%jsroot on   # interactive drawing in the notebook (needed on Binder)
import numpy as np

MATRICOLA = 1281521   # TODO: write your student ID number here (digits only)
assert MATRICOLA != 0, "Set MATRICOLA first: it seeds YOUR random toy data"

ROOT.RooRandom.randomGenerator().SetSeed(MATRICOLA)
ROOT.RooMsgService.instance().setGlobalKillBelow(ROOT.RooFit.WARNING)

def check(name, ok):
    """Minimal self-check: prints PASS/FAIL and stops the notebook on FAIL."""
    print(("PASS  " if ok else "FAIL  ") + name)
    assert ok, name

## B1 · Breit–Wigner fit of a binned mass spectrum
1. Import the histogram as a `RooDataHist`. 2. Fit a Breit–Wigner. 3. Add a Gaussian resolution and refit.
4. Compare the mass with the PDG value (≈ 5.367 GeV for the $B^0_s$).

In [3]:
f = ROOT.TFile.Open("../../examples/lhcb-b0-ppbar/data/b0_invariant_mass.root")
h = f.Get("massaB0")
h.SetDirectory(0)
f.Close()

mass = ROOT.RooRealVar("mass", "mass [GeV]", 5.0, 6.0)
dh   = ROOT.RooDataHist("dh", "dh", ROOT.RooArgList(mass), h)   # TODO: RooDataHist from h, with observable mass:  ROOT.RooDataHist("dh","dh", ROOT.RooArgList(mass), h)

mean_bw  = ROOT.RooRealVar("mean_bw", "mass", 5.36, 5.2, 5.5)
width_bw = ROOT.RooRealVar("width_bw", "width", 0.1, 0.001, 0.5)
bw       = ROOT.RooBreitWigner("bw","bw", mass, mean_bw, width_bw)   # TODO: RooBreitWigner(mass, mean_bw, width_bw)
res_bw   = bw.fitTo(dh, Save=True, PrintLevel=-1)   # TODO: binned ML fit (Save=True, PrintLevel=-1)

frame_b = mass.frame(Title="Breit-Wigner fit")
# TODO: plot dh, then bw
dh.plotOn(frame_b)
bw.plotOn(frame_b)
c_b = ROOT.TCanvas("c_b", "c_b", 800, 500)
frame_b.Draw()
c_b.Draw()
mean_bw.Print(); width_bw.Print()

RooRealVar::mean_bw = 5.36235 +/- 0.0026698  L(5.2 - 5.5) 
RooRealVar::width_bw = 0.0987366 +/- 0.00485722  L(0.001 - 0.5) 


In [4]:
check("BW fit converged", res_bw is not None and res_bw.status() == 0)
check("mass within 30 MeV of PDG (5.367 GeV)", abs(mean_bw.getVal() - 5.367) < 0.03)

PASS  BW fit converged
PASS  mass within 30 MeV of PDG (5.367 GeV)


### Add a Gaussian resolution
`RooVoigtian` is the analytic convolution Breit–Wigner ⊗ Gaussian.
Fix the resolution at σ_res = 30 MeV (it cannot be floated together with the width: why not?) and refit.

In [6]:
mean_v  = ROOT.RooRealVar("mean_v", "mass", 5.36, 5.2, 5.5)
width_v = ROOT.RooRealVar("width_v", "width", 0.1, 0.001, 0.5)
sig_res = ROOT.RooRealVar("sig_res", "resolution", 0.03)   # fixed value, no range
voigt   = ROOT.RooVoigtian("voigt", "voigt", mass, mean_v, width_v, sig_res)   # TODO: RooVoigtian(mass, mean_v, width_v, sig_res)
res_v   = voigt.fitTo(dh, Save=True, PrintLevel=-1)  # TODO: fit it to dh

frame_v = mass.frame(Title="Voigtian = BW (x) Gaussian")
# TODO: plot dh and voigt
dh.plotOn(frame_v)
voigt.plotOn(frame_v)
c_v = ROOT.TCanvas("c_v", "c_v", 800, 500)
frame_v.Draw()
c_v.Draw()
print("BW width:", width_bw.getVal(), " Voigtian BW width:", width_v.getVal())

BW width: 0.09873663553231045  Voigtian BW width: 0.07391015885593899


In [7]:
check("Voigtian fit converged", res_v is not None and res_v.status() == 0)

PASS  Voigtian fit converged


In [9]:
print("mass form fitting:", mean_v.getVal())

mass form fitting: 5.361924195594515


**Questions B1.** 

(a) How far is your mass from the PDG value, in units of the fit error?
 - $\Delta m / \sigma_{m} = 0.005/0.074 = 0.067$

(b) The natural width of the $B^0_s$ is of order 10⁻¹² GeV. Is the fitted width physical? What does it measure?
-  The fitted width ($\sim 50\text{--}70\text{ MeV}$) is dominated by experimental detector resolution; natural width ($\sim 10^{-12}\text{ GeV}$) is completely negligible.

(c) Why do width and resolution cannot be fitted together here?
- Breit–Wigner width and Gaussian resolution broaden the peak; floating both causes degenerate/highly correlated parameters that cannot be disentangled without external resolution calibration.

## B2 · The central limit theorem, and its tails
Sum of N independent uniform variables in [0,1]. The sum has mean N/2 and variance N/12.
1. Fit a Gaussian with RooFit to 10 000 sums (N = 10).
2. Count the fraction of 2 000 000 sums outside ±kσ (k = 1…4) and compare with the Gaussian expectation, for N = 2, 5, 10, 20.

In [11]:
import math
rng = np.random.default_rng(MATRICOLA)
NEVT = 2_000_000

def make_sums(N):
    s = np.zeros(NEVT)
    for _ in range(N):
        s += rng.random(NEVT)
    return s

def fit_gauss(sums, N, nfit=10000):
    """Fit a RooGaussian to the first nfit sums. Return (mean_var, sigma_var, result)."""
    # TODO: observable S in [0, N]; fill a RooDataSet with sums[:nfit]; fit a Gaussian; return the results
    S = ROOT.RooRealVar("S", "S", 0, N)
    mean_var = ROOT.RooRealVar("mean", "mean", N / 2, 0, N)
    sigma_var = ROOT.RooRealVar("sigma", "sigma", math.sqrt(N / 12), 0.01, N)
    gauss = ROOT.RooGaussian("gauss", "gauss", S, mean_var, sigma_var)
    data = ROOT.RooDataSet("data", "data", ROOT.RooArgSet(S))
    for v in sums[:nfit]:
        S.setVal(v)
        data.add(ROOT.RooArgSet(S))
    result = gauss.fitTo(data, Save=True, PrintLevel=-1)
    return (mean_var, sigma_var, result)
    

In [12]:
sums10 = make_sums(10)
out = fit_gauss(sums10, 10)
check("fit_gauss implemented", out is not None)
m, sg, res = out
print(f"N=10: mean = {m.getVal():.3f} +- {m.getError():.3f}  (expect 5),  sigma = {sg.getVal():.3f} +- {sg.getError():.3f}  (expect {math.sqrt(10/12):.3f})")
check("mean compatible with N/2", abs(m.getVal() - 5) < 4 * m.getError())
check("sigma compatible with sqrt(N/12)", abs(sg.getVal() - math.sqrt(10 / 12)) < 4 * sg.getError())

PASS  fit_gauss implemented
N=10: mean = 5.009 +- 0.009  (expect 5),  sigma = 0.909 +- 0.006  (expect 0.913)
PASS  mean compatible with N/2
PASS  sigma compatible with sqrt(N/12)


In [13]:
# Tail table: fraction outside +-k sigma, measured vs Gaussian
print(" N   k    measured      gauss      ratio")
for N in (2, 5, 10, 20):
    s = make_sums(N)
    z = (s - N / 2) / math.sqrt(N / 12)
    for k in (1, 2, 3, 4):
        # TODO: measured = fraction of |z| > k ; expected = math.erfc(k / math.sqrt(2)); print one line
        measured = np.mean(np.abs(z) > k)
        expected = math.erfc(k / math.sqrt(2))
        ratio = measured / expected if expected > 0 else 0.0
        print(f"{N:2d}  {k:2d}    {measured:.6f}    {expected:.6f}    {ratio:.3f}")
        pass

 N   k    measured      gauss      ratio
 2   1    0.350482    0.317311    1.105
 2   2    0.033921    0.045500    0.746
 2   3    0.000000    0.002700    0.000
 2   4    0.000000    0.000063    0.000
 5   1    0.327215    0.317311    1.031
 5   2    0.042996    0.045500    0.945
 5   3    0.000954    0.002700    0.353
 5   4    0.000000    0.000063    0.000
10   1    0.321583    0.317311    1.013
10   2    0.044436    0.045500    0.977
10   3    0.001920    0.002700    0.711
10   4    0.000010    0.000063    0.166
20   1    0.319334    0.317311    1.006
20   2    0.044813    0.045500    0.985
20   3    0.002343    0.002700    0.868
20   4    0.000036    0.000063    0.576


**Questions B2.** 

(a) For which (N, k) does the Gaussian tail probability fail by more than 20 %?
- For $k \ge 3$, the Gaussian approximation deviates significantly ($> 20%$) when $N \le 10$.

(b) For N = 2 the sum is triangular and has a hard edge: what happens beyond ±2.45σ?
- For $N=2$, the sum $S \in [0, 2]$ has support $|z| \le \frac{1}{\sqrt{2/12}} = \sqrt{6} \approx 2.45$; beyond $2.45\sigma$, the probability is strictly 0.

(c) What does this teach about quoting a "5σ" result from a Gaussian approximation?
- Quoting $5\sigma$ under CLT/Gaussian assumptions is invalid far into the tails where real distributions truncate or have non-Gaussian tails.